In [1]:
# ===========================================
# Libraries
# ===========================================

import numpy as np

In [2]:
# ===========================================
# Load evidence results for all 6 models
# ===========================================

EVBASE = "/home/brunowesley/projetos/MCMC-cosmo/Codes/Model-comparison/Bayesian-Evidence"

model_tags = ["LCDM", "wCDM", "CPL", "AB", "HS", "ST"]
model_k = {"LCDM": 6, "wCDM": 7, "CPL": 8, "AB": 7, "HS": 7, "ST": 7}

results = {}
for tag in model_tags:
    data = np.load(f"{EVBASE}/{tag}/evidence_{tag}.npz", allow_pickle=True)
    entry = {}
    for t, lnZ, sig, N in zip(data["tags"], data["lnZ"], data["sigma_lnZ"], data["N_used"]):
        entry[str(t)] = {"lnZ": float(lnZ), "sigma": float(sig), "N": int(N)}
    results[tag] = entry

for tag in model_tags:
    b = results[tag]["Baseline"]
    f = results[tag]["Baseline + FRB"]
    print(f"{tag:6s}  Baseline: lnZ={b['lnZ']:.3f}+/-{b['sigma']:.4f}   "
          f"Baseline+FRB: lnZ={f['lnZ']:.3f}+/-{f['sigma']:.4f}")

LCDM    Baseline: lnZ=-747.483+/-0.0044   Baseline+FRB: lnZ=-1401.920+/-0.0043
wCDM    Baseline: lnZ=-746.777+/-0.0051   Baseline+FRB: lnZ=-1400.303+/-0.0050
CPL     Baseline: lnZ=-747.632+/-0.0095   Baseline+FRB: lnZ=-1401.825+/-0.0041
AB      Baseline: lnZ=-747.765+/-0.0110   Baseline+FRB: lnZ=-1402.305+/-0.0062
HS      Baseline: lnZ=-745.133+/-0.0044   Baseline+FRB: lnZ=-1398.717+/-0.0039
ST      Baseline: lnZ=-744.210+/-0.0118   Baseline+FRB: lnZ=-1397.899+/-0.0106


In [3]:
# ===========================================
# Bayes factors relative to LCDM
# ===========================================
# ln BF(model) = ln Z(model) - ln Z(LCDM), for each dataset combination.
# Uncertainties combine in quadrature (independent evidence estimates).

def jeffreys_label(lnbf):
    a = abs(lnbf)
    if a < 1.0:
        strength = "not worth more than a bare mention"
    elif a < 2.5:
        strength = "substantial"
    elif a < 5.0:
        strength = "strong"
    else:
        strength = "decisive"
    direction = "for alternative" if lnbf > 0 else "for LCDM"
    return f"{strength} ({direction})" if a >= 1.0 else strength

bayes_factors = {}
for dataset in ["Baseline", "Baseline + FRB"]:
    lnZ_ref = results["LCDM"][dataset]["lnZ"]
    sig_ref = results["LCDM"][dataset]["sigma"]
    bayes_factors[dataset] = {}
    for tag in model_tags[1:]:
        lnZ = results[tag][dataset]["lnZ"]
        sig = results[tag][dataset]["sigma"]
        lnbf = lnZ - lnZ_ref
        sig_bf = np.sqrt(sig**2 + sig_ref**2)
        bayes_factors[dataset][tag] = {"lnBF": lnbf, "sigma": sig_bf,
                                        "label": jeffreys_label(lnbf)}

for dataset in ["Baseline", "Baseline + FRB"]:
    print(f"\n--- {dataset} (relative to LCDM) ---")
    for tag in model_tags[1:]:
        r = bayes_factors[dataset][tag]
        print(f"  {tag:6s}: ln BF = {r['lnBF']:+.3f} +/- {r['sigma']:.4f}   [{r['label']}]")


--- Baseline (relative to LCDM) ---
  wCDM  : ln BF = +0.706 +/- 0.0068   [not worth more than a bare mention]
  CPL   : ln BF = -0.150 +/- 0.0105   [not worth more than a bare mention]
  AB    : ln BF = -0.282 +/- 0.0118   [not worth more than a bare mention]
  HS    : ln BF = +2.350 +/- 0.0062   [substantial (for alternative)]
  ST    : ln BF = +3.272 +/- 0.0126   [strong (for alternative)]

--- Baseline + FRB (relative to LCDM) ---
  wCDM  : ln BF = +1.617 +/- 0.0066   [substantial (for alternative)]
  CPL   : ln BF = +0.095 +/- 0.0059   [not worth more than a bare mention]
  AB    : ln BF = -0.385 +/- 0.0076   [not worth more than a bare mention]
  HS    : ln BF = +3.203 +/- 0.0058   [strong (for alternative)]
  ST    : ln BF = +4.021 +/- 0.0115   [strong (for alternative)]


In [4]:
# ===========================================
# Comparison with the BIC-based ln BF approximation
# ===========================================
# The existing Model_comparison.ipynb (Codes/Model-comparison/AIC-BIC-LRT/) reports an
# approximate ln BF via the standard Schwarz/Laplace relation
# ln BF ~ -Delta BIC / 2, evaluated for Baseline + FRB only. Values below are
# taken directly from model_comparison.txt for comparison with our k-NN
# evidence-based ln BF (Baseline + FRB).

bic_approx_lnBF = {
    "wCDM": 1.5858,
    "CPL": -1.9316,
    "AB": -3.8588,
    "HS": 1.3525,
    "ST": 1.3619,
}

print("\n" + "=" * 78)
header = "{:6s} {:>22s} {:>20s} {:>10s}".format(
    "Model", "lnBF (k-NN evidence)", "lnBF (BIC approx.)", "Delta")
print(header)
print("=" * 78)
for tag in model_tags[1:]:
    r = bayes_factors["Baseline + FRB"][tag]
    approx = bic_approx_lnBF[tag]
    diff = r["lnBF"] - approx
    print(f"{tag:6s} {r['lnBF']:+18.3f}    {approx:+16.4f}    {diff:+10.3f}")
print("=" * 78)


Model    lnBF (k-NN evidence)   lnBF (BIC approx.)      Delta
wCDM               +1.617             +1.5858        +0.031
CPL                +0.095             -1.9316        +2.026
AB                 -0.385             -3.8588        +3.474
HS                 +3.203             +1.3525        +1.851
ST                 +4.021             +1.3619        +2.659


In [5]:
# ===========================================
# Save the final summary (evidence + Bayes factors + comparison)
# ===========================================

outfile = "BayesFactors_summary.txt"
with open(outfile, "w", encoding="utf-8") as f:
    f.write("=" * 78 + "\n")
    f.write(f"{'BAYESIAN EVIDENCE AND BAYES FACTORS (k=1 NN, Heavens et al. 2017)':^78}\n")
    f.write(f"{'Reference: LCDM':^78}\n")
    f.write("=" * 78 + "\n\n")

    f.write("--- Log-evidence ln Z ---\n")
    for tag in model_tags:
        b = results[tag]["Baseline"]
        fr = results[tag]["Baseline + FRB"]
        f.write(f"{tag:6s}  Baseline: {b['lnZ']:.3f} +/- {b['sigma']:.4f}   "
                f"Baseline+FRB: {fr['lnZ']:.3f} +/- {fr['sigma']:.4f}\n")

    for dataset in ["Baseline", "Baseline + FRB"]:
        f.write(f"\n--- ln BF vs LCDM ({dataset}) ---\n")
        for tag in model_tags[1:]:
            r = bayes_factors[dataset][tag]
            f.write(f"  {tag:6s}: ln BF = {r['lnBF']:+.3f} +/- {r['sigma']:.4f}   [{r['label']}]\n")

    f.write("\n--- Comparison with BIC-based ln BF approximation (Baseline + FRB) ---\n")
    for tag in model_tags[1:]:
        r = bayes_factors["Baseline + FRB"][tag]
        approx = bic_approx_lnBF[tag]
        f.write(f"  {tag:6s}: k-NN={r['lnBF']:+.3f}   BIC-approx={approx:+.4f}   "
                f"Delta={r['lnBF']-approx:+.3f}\n")
    f.write("\n" + "=" * 78 + "\n")

np.savez("BayesFactors_summary.npz",
         model_tags=model_tags[1:],
         lnBF_baseline=[bayes_factors["Baseline"][t]["lnBF"] for t in model_tags[1:]],
         sigma_baseline=[bayes_factors["Baseline"][t]["sigma"] for t in model_tags[1:]],
         lnBF_frb=[bayes_factors["Baseline + FRB"][t]["lnBF"] for t in model_tags[1:]],
         sigma_frb=[bayes_factors["Baseline + FRB"][t]["sigma"] for t in model_tags[1:]])

print("Saved: BayesFactors_summary.txt, BayesFactors_summary.npz")

Saved: BayesFactors_summary.txt, BayesFactors_summary.npz
